# TripMind fare estimator (LightGBM regression)

Learns typical Indian domestic economy and business fares, so TripMind can tell a traveller
whether a fare is **below typical, typical or above typical** for that route length, timing and
stops. It never changes a price; it only adds context.

**Run on Kaggle:** *Add data* → dataset
[Flight Price Prediction](https://www.kaggle.com/datasets/shubhambathwal/flight-price-prediction)
(file `Clean_Dataset.csv`, about 300,000 flights between six Indian metros), then *Run all*.
CPU is enough.

**Hand back:** `model.txt` and `model_card.json` from `/kaggle/working` into
`models/fare_estimator/artifacts/`. Features match `FARE_ESTIMATOR_FEATURES` in
`backend/app/modules/ml/features.py`. TripMind only scores routes up to 2,000 km, the range
this data covers.

In [ ]:
import glob, math
import numpy as np, pandas as pd, lightgbm
path = glob.glob("/kaggle/input/**/Clean_Dataset.csv", recursive=True)[0]
df = pd.read_csv(path)
print(df.shape); df.head()

Distances use the same airport coordinates as TripMind's catalog, so the feature means the same thing at serving time.

In [ ]:
AIRPORTS = {"Delhi": (28.5562, 77.1000), "Mumbai": (19.0896, 72.8656), "Bangalore": (13.1986, 77.7066),
            "Kolkata": (22.6547, 88.4467), "Hyderabad": (17.2403, 78.4294), "Chennai": (12.9941, 80.1709)}
def km(a, b):
    lat1, lon1, lat2, lon2 = map(math.radians, (*a, *b))
    h = math.sin((lat2 - lat1) / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * math.asin(math.sqrt(h))
HOURS = {"Late_Night": 2, "Early_Morning": 6, "Morning": 10, "Afternoon": 14, "Evening": 18, "Night": 22}
STOPS = {"zero": 0, "one": 1, "two_or_more": 2}
FEATURES = ("distance_km", "duration_minutes", "stops", "days_left", "departure_hour", "business")
X = pd.DataFrame({
    "distance_km": [km(AIRPORTS[s], AIRPORTS[d]) for s, d in zip(df.source_city, df.destination_city)],
    "duration_minutes": df.duration * 60,
    "stops": df.stops.map(STOPS),
    "days_left": df.days_left.clip(1, 50),
    "departure_hour": df.departure_time.map(HOURS),
    "business": (df["class"] == "Business").astype(float),
})[list(FEATURES)]
y = np.log(df.price)
print(X.describe().T[["min", "max"]])

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=7)
model = lightgbm.train({"objective": "regression_l1", "learning_rate": 0.05, "num_leaves": 63,
                        "min_data_in_leaf": 100, "verbose": -1},
                       lightgbm.Dataset(X_train, y_train), 1500,
                       valid_sets=[lightgbm.Dataset(X_test, y_test)],
                       callbacks=[lightgbm.early_stopping(100), lightgbm.log_evaluation(100)])
pred, actual = np.exp(model.predict(X_test)), np.exp(y_test)
# Baseline: median fare for the same cabin and distance band.
band = pd.cut(X_test.distance_km, 8)
baseline = pd.Series(actual.values, index=X_test.index).groupby([X_test.business, band], observed=True).transform("median")
metrics = {"mape_model": float(np.mean(np.abs(pred - actual) / actual)),
           "mape_median_baseline": float(np.mean(np.abs(baseline - actual) / actual)),
           "mae_inr_model": float(np.mean(np.abs(pred - actual))), "rows": float(len(df))}
print(metrics)
assert metrics["mape_model"] < metrics["mape_median_baseline"], "model does not beat the baseline: do not ship"

In [ ]:
import json, datetime, lightgbm
def write_card(name, features, dataset, metrics, notes):
    version = datetime.date.today().strftime("%Y.%m.%d")
    card = {"name": name, "version": version,
            "trained_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "framework": "lightgbm", "framework_version": lightgbm.__version__,
            "dataset": dataset, "features": list(features), "metrics": metrics, "notes": notes}
    with open("/kaggle/working/model_card.json", "w") as f:
        json.dump(card, f, indent=2)
    print(json.dumps(card, indent=2))

    model.save_model("/kaggle/working/model.txt", num_iteration=model.best_iteration)
    write_card("fare_estimator", FEATURES, "Flight Price Prediction (Kaggle), Indian domestic fares between six metros",
               metrics, "Predicts log(fare in INR) per adult. Airline identity is deliberately not a feature.")